# Step 1 - Imports

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.linear_model import Lasso
from xgboost import XGBRegressor

print("Imports OK")

Imports OK


# Step 2- Configuration

In [3]:
split_dir = Path("../data/split")

nintervals     = 96
lasso_max_iter = 10000
random_state   = 42
n_train        = 100
windows        = [100, 84, 56, 28]
target         = "price_eur_mwh"

# ablated feature set: cross-granularity features removed
features = [
    "price_lag1d",
    "price_lag7d",
    "wind_mwh",
    "solar_mwh",
    "load_mwh",
]

# best configs from the thesis
lear_alpha = 0.01   # best alpha from LEAR regularisation sensitivity (Table 4.5)

xgb_params = dict(
    n_estimators     = 444,
    learning_rate    = 0.020834136992346326,
    max_depth        = 4,
    min_child_weight = 6,
    subsample        = 0.6646124229308177,
    colsample_bytree = 0.6061035603483517,
    gamma            = 0.008910429703573388,
    reg_alpha        = 0.14968119527801219,
    reg_lambda       = 0.26277200687034197,
    random_state     = random_state,
)

print("Features (5):", features)

Features (5): ['price_lag1d', 'price_lag7d', 'wind_mwh', 'solar_mwh', 'load_mwh']


# Step 3 - Load data

In [4]:
df = pd.read_csv(split_dir / "feature_matrix_clean.csv", parse_dates=["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

train_days = [pd.Timestamp(d).date() for d in pd.read_csv(split_dir / "train_days.csv")["date"]]
test_days  = [pd.Timestamp(d).date() for d in pd.read_csv(split_dir / "test_days.csv")["date"]]

n_test = len(test_days)
X_all  = df[features].values
y_all  = df[target].values
y_test_orig = y_all[n_train * nintervals:].reshape(n_test, nintervals)

print(f"X_all: {X_all.shape}  (should be (13824, 5))")
print(f"Train: {n_train} days | Test: {n_test} days")
print(f"Test period: {test_days[0]} -> {test_days[-1]}")

X_all: (13824, 5)  (should be (13824, 5))
Train: 100 days | Test: 44 days
Test period: 2026-01-16 -> 2026-02-28


# Step 4 - Helper+ run function

In [5]:
def fit_and_predict(model_kind, row_start, row_end, row_pred, nintervals=nintervals):
    X_tr = X_all[row_start:row_end, :]
    y_tr = y_all[row_start:row_end]

    mean_X, std_X = X_tr.mean(axis=0), X_tr.std(axis=0)
    std_X[std_X == 0] = 1.0
    mean_y, std_y = y_tr.mean(), y_tr.std()
    if std_y == 0:
        std_y = 1.0

    X_tr_sc = (X_tr - mean_X) / std_X
    y_tr_sc = (y_tr - mean_y) / std_y
    X_te_sc = (X_all[row_pred:row_pred + nintervals, :] - mean_X) / std_X

    if model_kind == "lear":
        model = Lasso(alpha=lear_alpha, max_iter=lasso_max_iter, fit_intercept=True)
    else:
        model = XGBRegressor(**xgb_params)
    model.fit(X_tr_sc, y_tr_sc)
    pred_sc = model.predict(X_te_sc)
    return pred_sc * std_y + mean_y


def run_model(model_kind):
    all_preds = np.zeros((n_test, nintervals))
    for i in range(n_test):
        forecast_day_idx = n_train + i
        row_pred = forecast_day_idx * nintervals
        window_preds = []
        for w in windows:
            row_end   = forecast_day_idx * nintervals
            row_start = max(forecast_day_idx - w, 0) * nintervals
            window_preds.append(fit_and_predict(model_kind, row_start, row_end, row_pred))
        all_preds[i, :] = np.mean(window_preds, axis=0)
    errors = y_test_orig - all_preds
    mae  = float(np.mean(np.abs(errors)))
    rmse = float(np.sqrt(np.mean(errors ** 2)))
    return mae, rmse

print("Helpers defined")

Helpers defined


# Step 5 - Run both models (LEAR - XGBoost)

In [6]:
print("Running LEAR (5 feat) ...")
lear_mae, lear_rmse = run_model("lear")
print(f"  LEAR    -> MAE {lear_mae:.2f}  RMSE {lear_rmse:.2f}")

print("Running XGBoost (5 feat) ...")
xgb_mae, xgb_rmse = run_model("xgb")
print(f"  XGBoost -> MAE {xgb_mae:.2f}  RMSE {xgb_rmse:.2f}")

Running LEAR (5 feat) ...
  LEAR    -> MAE 11.89  RMSE 16.01
Running XGBoost (5 feat) ...
  XGBoost -> MAE 11.33  RMSE 16.62


# Step 6- Comparison 

In [7]:
# Thesis 7-feature results (fixed)
lear_full_mae, lear_full_rmse = 12.06, 16.08
xgb_full_mae,  xgb_full_rmse  = 11.55, 16.27

print("CROSS-GRANULARITY ABLATION RESULT")
print()
print(f"{'Model':<10}{'Features':<16}{'MAE':>8}{'RMSE':>9}")
print()
print(f"{'LEAR':<10}{'Full (7)':<16}{lear_full_mae:>8.2f}{lear_full_rmse:>9.2f}")
print(f"{'LEAR':<10}{'Ablated (5)':<16}{lear_mae:>8.2f}{lear_rmse:>9.2f}")
print()
print(f"{'XGBoost':<10}{'Full (7)':<16}{xgb_full_mae:>8.2f}{xgb_full_rmse:>9.2f}")
print(f"{'XGBoost':<10}{'Ablated (5)':<16}{xgb_mae:>8.2f}{xgb_rmse:>9.2f}")
print()

def report(name, full, abl):
    diff = abl - full
    pct  = diff / full * 100
    if abs(diff) < 0.05:
        v = "negligible effect"
    elif diff > 0:
        v = f"features HELP (removing worsens MAE by {diff:.2f}, {pct:+.1f}%)"
    else:
        v = f"features do not help (removing improves MAE by {-diff:.2f}, {pct:+.1f}%)"
    print(f"  {name}: {full:.2f} -> {abl:.2f}  =>  {v}")

print("Effect of removing cross-granularity features:")
report("LEAR   ", lear_full_mae, lear_mae)
report("XGBoost", xgb_full_mae, xgb_mae)

CROSS-GRANULARITY ABLATION RESULT

Model     Features             MAE     RMSE

LEAR      Full (7)           12.06    16.08
LEAR      Ablated (5)        11.89    16.01

XGBoost   Full (7)           11.55    16.27
XGBoost   Ablated (5)        11.33    16.62

Effect of removing cross-granularity features:
  LEAR   : 12.06 -> 11.89  =>  features do not help (removing improves MAE by 0.17, -1.4%)
  XGBoost: 11.55 -> 11.33  =>  features do not help (removing improves MAE by 0.22, -1.9%)


# Step 7 - LSTM 

In [9]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cpu")

# LSTM hyperparameters — identical to LSTM.ipynb
seq_days      = 7
hidden_size   = 64      # baseline (best) LSTM config
num_layers    = 2
dropout       = 0.2
learning_rate = 0.001
max_epochs    = 50
batch_size    = 16
patience      = 10
random_state  = 42

features_full_lstm = [
    "price_lag1d", "price_lag7d",
    "price_hourly_lag1d", "price_hourly_lag7d",
    "wind_mwh", "solar_mwh", "load_mwh",
]
features_ablated_lstm = [
    "price_lag1d", "price_lag7d",
    "wind_mwh", "solar_mwh", "load_mwh",
]

print(f"PyTorch {torch.__version__} | device {device}")
print("LSTM full   :", features_full_lstm)
print("LSTM ablated:", features_ablated_lstm)

PyTorch 2.11.0+cpu | device cpu
LSTM full   : ['price_lag1d', 'price_lag7d', 'price_hourly_lag1d', 'price_hourly_lag7d', 'wind_mwh', 'solar_mwh', 'load_mwh']
LSTM ablated: ['price_lag1d', 'price_lag7d', 'wind_mwh', 'solar_mwh', 'load_mwh']


# Step 8 - model class and run function

In [10]:
class LSTMForecaster(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size, dropout):
        super(LSTMForecaster, self).__init__()
        self.lstm = nn.LSTM(
            input_size=input_size, hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last_step = lstm_out[:, -1, :]
        return self.fc(last_step)


def build_sequences(X_norm, y_norm, seq_days):
    n_days = len(y_norm)
    X_seq, y_seq = [], []
    for d in range(seq_days, n_days):
        seq = X_norm[d - seq_days:d]
        X_seq.append(seq.reshape(seq_days, -1))
        y_seq.append(y_norm[d])
    return np.array(X_seq), np.array(y_seq)


def run_lstm(feature_list):
    """Run the full expanding-window LSTM for one feature set, return MAE/RMSE."""
    nfeatures   = len(feature_list)
    input_size  = nintervals * nfeatures
    output_size = nintervals

    # day-level arrays for this feature set
    all_days = sorted(df["timestamp"].dt.date.unique())
    n_total  = len(all_days)
    X_days = np.zeros((n_total, nintervals, nfeatures))
    y_days = np.zeros((n_total, nintervals))
    for i, day in enumerate(all_days):
        day_df = df[df["timestamp"].dt.date == day].sort_values("timestamp")
        X_days[i] = day_df[feature_list].values
        y_days[i] = day_df[target].values

    y_test_orig_lstm = y_days[n_train:].copy()
    lstm_predictions = np.zeros((n_test, nintervals))

    for i in range(n_test):
        forecast_day_idx = n_train + i

        X_tr_raw = X_days[:forecast_day_idx]
        y_tr_raw = y_days[:forecast_day_idx]
        offset_X = X_tr_raw.reshape(-1, nfeatures).mean(axis=0)
        scale_X  = X_tr_raw.reshape(-1, nfeatures).std(axis=0)
        scale_X[scale_X == 0] = 1.0
        offset_y = float(y_tr_raw.mean())
        scale_y  = float(y_tr_raw.std())
        if scale_y == 0:
            scale_y = 1.0

        X_norm_i = (X_days[:forecast_day_idx + 1] - offset_X) / scale_X
        y_norm_i = (y_days[:forecast_day_idx]     - offset_y) / scale_y

        X_seq_i, y_seq_i = build_sequences(X_norm_i, y_norm_i, seq_days)

        n_val = max(1, int(0.1 * len(X_seq_i)))
        X_val = torch.tensor(X_seq_i[-n_val:], dtype=torch.float32).to(device)
        y_val = torch.tensor(y_seq_i[-n_val:], dtype=torch.float32).to(device)
        X_tr  = torch.tensor(X_seq_i[:-n_val], dtype=torch.float32).to(device)
        y_tr  = torch.tensor(y_seq_i[:-n_val], dtype=torch.float32).to(device)

        g = torch.Generator(); g.manual_seed(random_state)
        train_loader = DataLoader(TensorDataset(X_tr, y_tr),
                                  batch_size=batch_size, shuffle=True, generator=g)

        torch.manual_seed(random_state)
        model = LSTMForecaster(input_size, hidden_size, num_layers,
                               output_size, dropout).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
        criterion = nn.MSELoss()

        best_val_loss, patience_counter, best_state = float("inf"), 0, None
        for epoch in range(max_epochs):
            model.train()
            for X_batch, y_batch in train_loader:
                optimizer.zero_grad()
                loss = criterion(model(X_batch), y_batch)
                loss.backward()
                optimizer.step()
            model.eval()
            with torch.no_grad():
                val_loss = criterion(model(X_val), y_val).item()
            if val_loss < best_val_loss:
                best_val_loss, patience_counter = val_loss, 0
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    break

        model.load_state_dict(best_state)
        model.eval()
        X_pred = X_norm_i[forecast_day_idx - seq_days: forecast_day_idx]
        X_pred = torch.tensor(X_pred.reshape(1, seq_days, -1), dtype=torch.float32).to(device)
        with torch.no_grad():
            pred_sc = model(X_pred).cpu().numpy().squeeze()
        lstm_predictions[i] = pred_sc * scale_y + offset_y

        if (i + 1) % 10 == 0 or i == 0:
            print(f"    Day {i+1:>2d}/44 (trained on {forecast_day_idx} days)")

    errors = y_test_orig_lstm - lstm_predictions
    mae  = float(np.mean(np.abs(errors)))
    rmse = float(np.sqrt(np.mean(errors ** 2)))
    return mae, rmse

print("LSTM run function defined")

LSTM run function defined


# Step 9 - run 5 features

In [13]:
# known 7-feature LSTM baseline (from LSTM.ipynb, hidden_size=64)
lstm_full_mae, lstm_full_rmse = 20.64, 28.51

print("Running LSTM (ablate")
lstm_abl_mae, lstm_abl_rmse = run_lstm(features_ablated_lstm)
print(f"  LSTM ablated -> MAE {lstm_abl_mae:.2f}  RMSE {lstm_abl_rmse:.2f}")
print(f"  (LSTM full 7-feat, from thesis: MAE {lstm_full_mae:.2f}  RMSE {lstm_full_rmse:.2f})")

Running LSTM (ablate
    Day  1/44 (trained on 100 days)
    Day 10/44 (trained on 109 days)
    Day 20/44 (trained on 119 days)
    Day 30/44 (trained on 129 days)
    Day 40/44 (trained on 139 days)
  LSTM ablated -> MAE 20.26  RMSE 27.80
  (LSTM full 7-feat, from thesis: MAE 20.64  RMSE 28.51)


# Step 10 - Comparison

In [14]:
print("CROSS-GRANULARITY ABLATION - ALL THREE MODELS")
print("(quarter-hourly data, same period/windows/hyperparameters)")
print()
print(f"{'Model':<10}{'Features':<16}{'MAE':>8}{'RMSE':>9}")
print()
print(f"{'LEAR':<10}{'Full (7)':<16}{lear_full_mae:>8.2f}{lear_full_rmse:>9.2f}")
print(f"{'LEAR':<10}{'Ablated (5)':<16}{lear_mae:>8.2f}{lear_rmse:>9.2f}")
print()
print(f"{'XGBoost':<10}{'Full (7)':<16}{xgb_full_mae:>8.2f}{xgb_full_rmse:>9.2f}")
print(f"{'XGBoost':<10}{'Ablated (5)':<16}{xgb_mae:>8.2f}{xgb_rmse:>9.2f}")
print()
print(f"{'LSTM':<10}{'Full (7)':<16}{lstm_full_mae:>8.2f}{lstm_full_rmse:>9.2f}")
print(f"{'LSTM':<10}{'Ablated (5)':<16}{lstm_abl_mae:>8.2f}{lstm_abl_rmse:>9.2f}")
print()

def report(name, full, abl):
    diff = abl - full
    pct  = diff / full * 100
    if abs(diff) < 0.05:
        v = "negligible effect"
    elif diff > 0:
        v = f"features HELP (removing worsens MAE by {diff:.2f}, {pct:+.1f}%)"
    else:
        v = f"features do not help (removing improves MAE by {-diff:.2f}, {pct:+.1f}%)"
    print(f"  {name}: {full:.2f} -> {abl:.2f}  =>  {v}")

print("Effect of removing cross-granularity features:")
report("LEAR   ", lear_full_mae, lear_mae)
report("XGBoost", xgb_full_mae, xgb_mae)
report("LSTM   ", lstm_full_mae, lstm_abl_mae)

CROSS-GRANULARITY ABLATION - ALL THREE MODELS
(quarter-hourly data, same period/windows/hyperparameters)

Model     Features             MAE     RMSE

LEAR      Full (7)           12.06    16.08
LEAR      Ablated (5)        11.89    16.01

XGBoost   Full (7)           11.55    16.27
XGBoost   Ablated (5)        11.33    16.62

LSTM      Full (7)           20.64    28.51
LSTM      Ablated (5)        20.26    27.80

Effect of removing cross-granularity features:
  LEAR   : 12.06 -> 11.89  =>  features do not help (removing improves MAE by 0.17, -1.4%)
  XGBoost: 11.55 -> 11.33  =>  features do not help (removing improves MAE by 0.22, -1.9%)
  LSTM   : 20.64 -> 20.26  =>  features do not help (removing improves MAE by 0.38, -1.9%)
